In [8]:
import os

PATH = '../../data/../data/SFPI/'

os.listdir(PATH)

['.DS_Store', 'Images', 'labels', 'Annotations', 'data.yaml']

In [9]:
import yaml

# Path to your data.yaml
yaml_path = f"{PATH}/data.yaml"

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Pulling the class names
classes = data.get('names')

print(f"Total classes: {len(classes)}")
print("Labels:", classes)

Total classes: 8
Labels: ['armchair', 'bed', 'door', 'sink', 'sofa', 'table', 'tub', 'window']


In [10]:
import json
from pathlib import Path
from collections import defaultdict

SFPI = Path("../../data/SFPI")

# Merge numbered variants into a single class
MERGE = {
    "armchair": "armchair",
    "bed":      "bed",
    "door1":    "door",  "door2":   "door",
    "sink1":    "sink",  "sink2":   "sink",  "sink3": "sink", "sink4": "sink",
    "sofa1":    "sofa",  "sofa2":   "sofa",
    "table1":   "table", "table2":  "table", "table3": "table",
    "tub":      "tub",
    "window1":  "window","window2": "window",
}
MERGED_CLASSES = ["armchair", "bed", "door", "sink", "sofa", "table", "tub", "window"]
cls2idx = {c: i for i, c in enumerate(MERGED_CLASSES)}

for split in ("train", "val", "test"):
    ann_path = SFPI / "Annotations" / f"{split}_annotation.json"
    lbl_dir  = SFPI / "labels" / split
    lbl_dir.mkdir(parents=True, exist_ok=True)

    with open(ann_path) as f:
        coco = json.load(f)

    id2img  = {img["id"]: img for img in coco["images"]}
    id2name = {c["id"]: c["name"] for c in coco["categories"]}

    img2anns = defaultdict(list)
    for ann in coco["annotations"]:
        img2anns[ann["image_id"]].append(ann)

    for img_id, img_meta in id2img.items():
        W, H = img_meta["width"], img_meta["height"]
        stem = Path(img_meta["file_name"]).stem
        lines = []
        for ann in img2anns[img_id]:
            x, y, w, h = ann["bbox"]
            cx = (x + w / 2) / W
            cy = (y + h / 2) / H
            nw = w / W
            nh = h / H
            orig_name = id2name[ann["category_id"]]
            cls = cls2idx[MERGE[orig_name]]
            lines.append(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        (lbl_dir / f"{stem}.txt").write_text("\n".join(lines))

    print(f"{split}: wrote {len(id2img)} label files → {lbl_dir}")


train: wrote 7000 label files → ../../data/SFPI/labels/train
val: wrote 1500 label files → ../../data/SFPI/labels/val
test: wrote 1500 label files → ../../data/SFPI/labels/test


In [11]:
from pathlib import Path

image_dir = Path(f"{PATH}/train/images")
label_dir = Path(f"{PATH}/train/labels")

backgrounds = []

for img in image_dir.glob("*"):
    label = label_dir / f"{img.stem}.txt"

    if not label.exists() or label.stat().st_size == 0:
        backgrounds.append(img)

print(f"Background images: {len(backgrounds)}")
print(backgrounds[:10])

Background images: 0
[]


In [12]:
from pathlib import Path
import os

image_dir = Path(f"{PATH}/train/images")
label_dir = Path(f"{PATH}/train/labels")

removed = 0

for img in image_dir.glob("*"):
    label = label_dir / f"{img.stem}.txt"

    if not label.exists() or label.stat().st_size == 0:
        os.remove(img)

        if label.exists():
            os.remove(label)

        removed += 1

print(f"Removed {removed} background images")

Removed 0 background images


In [ ]:
from ultralytics import YOLO
import os
project=os.path.join(os.getcwd(), "../../runs/detect")  # adjust depth to repo root

# 1. Load the YOLO26 Nano model (pretrained on COCO)
# Using the .pt file ensures we are fine-tuning, not training from scratch
model = YOLO('yolo26n.pt')

# 2. Fine-tune the model
# dataset.location was defined when you ran version.download("yolo26")
results = model.train(
    data=f"{PATH}/data.yaml",
    project=project,
    epochs=2,
    imgsz=1024,
    batch=-1,          # high imgsz eats memory fast; 4-8 is realistic at 1024
    workers=4,
    cache="ram",
    device="mps",
    amp=True,
    rect=True,        # batches images of similar aspect ratio → less padding waste
    plots=True,
    patience=5,       # stop if no val improvement for 5 consecutive epochs
)

Ultralytics 8.4.47 🚀 Python-3.12.0 torch-2.11.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../data/../data/SFPI//data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, perspective=0.0,

KeyboardInterrupt: 

In [14]:
# Validate the model's performance on the validation set
metrics = model.val()

# Export for deployment (e.g., to ONNX for web/mobile use)
model.export(format='onnx')

Ultralytics 8.4.47 🚀 Python-3.12.0 torch-2.11.0 CPU (Apple M5)
YOLO26n summary (fused): 122 layers, 2,376,396 parameters, 1,764 gradients, 5.2 GFLOPs


FileNotFoundError: '/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml' does not exist

In [16]:
# model = YOLO("../../runs/detect/train/weights/best.pt")

# Run inference
results = model("../../data/PNG/no_text/IIa_AP3601.png")

# Show results
results[0].show()

# Save results
results[0].save(filename="result.jpg")


image 1/1 /Users/rx0/Desktop/FoundationOfAIML/2/generational-bullshit/notebooks/model/../../data/PNG/no_text/IIa_AP3601.png: 640x480 (no detections), 34.2ms
Speed: 1.9ms preprocess, 34.2ms inference, 0.1ms postprocess per image at shape (1, 3, 640, 480)


'result.jpg'

In [ ]:
import cv2
# Load and resize image
image = cv2.imread("../../data/PNG/no_text/IIa_A01001.png")
image_resized = cv2.resize(image, (640, 640))

# Run inference
results = model(image_resized)

# Show results
results[0].show()

# Save annotated result
results[0].save(filename="result.jpg")


0: 1024x1024 9 doors, 94.5ms
Speed: 3.7ms preprocess, 94.5ms inference, 0.1ms postprocess per image at shape (1, 3, 1024, 1024)


'result.jpg'